# 🧪 Práctica: Explorando Storages en Big Data con Docker

## Objetivo
Comprender las diferencias entre almacenamiento en memoria, almacenamiento en disco estructurado, y almacenamiento orientado a documentos, usando contenedores Docker.

## Componentes del Lab

| Tipo de Storage | Tecnología Dockerizada | Uso en Big Data |
|-----------------|------------------------|------------------|
| En memoria | Redis | Caching, procesamiento rápido |
| Estructurado en disco | PostgreSQL | ETL, almacenamiento relacional |
| Documental | MongoDB | Datos semiestructurados, flexibilidad |

---
## 📦 Instalación de dependencias

In [1]:
%pip install redis psycopg2-binary pymongo matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 11.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 917.4/917.4 kB 10.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pymongo]m3/4 [pymongo]n]

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


---
## 🔴 REDIS (Almacenamiento en Memoria)

### Simular sesiones de usuario

In [2]:
import redis

r = redis.Redis(host='redis', port=6379, decode_responses=True)

# Simular sesiones de usuario
r.set("user:1001", "active")
r.set("user:1002", "inactive")
r.set("user:1003", "active")
r.set("user:1004", "inactive")
r.set("user:1005", "active")
r.set("user:1006", "inactive")
r.set("user:1007", "active")
r.set("user:1008", "inactive")
r.set("user:1009", "active")
r.set("user:10010", "inactive")

# Recuperar datos
print("Estado del usuario 1001:", r.get("user:1001"))

ConnectionError: Error 8 connecting to redis:6379. nodename nor servname provided, or not known.

---
## 🐘 POSTGRESQL (Almacenamiento Relacional)

### Conectarse y crear la tabla ventas

In [ ]:
import psycopg2

conn = psycopg2.connect(
    host="postgres",
    database="testdb",
    user="admin",
    password="admin"
)
cur = conn.cursor()

# Crear tabla ventas
cur.execute("""
CREATE TABLE IF NOT EXISTS ventas (
    id SERIAL PRIMARY KEY,
    producto TEXT,
    cantidad INT,
    fecha DATE
)
""")
conn.commit()

# Insertar datos
cur.execute("INSERT INTO ventas (producto, cantidad, fecha) VALUES (%s, %s, %s)",
            ("Laptop", 5, "2025-08-14"))
conn.commit()

# Consultar
cur.execute("SELECT * FROM ventas")
print("Ventas registradas:")
print(cur.fetchall())

---
## 🍃 MONGODB (Almacenamiento Documental)

### Conectarse, listar bases de datos e insertar documento

In [ ]:
from pymongo import MongoClient

client = MongoClient("mongodb://mongodb:27017/")

# Mostrar bases de datos existentes
print("Bases de datos disponibles:", client.list_database_names())

db = client.bigdata

# Insertar documento en colección "alumnos"
db.alumnos.insert_one({
    "nombre": "TU NOMBRE COMPLETO",
    "intereses": ["Big Data", "Docker"],
    "asistencia": True
})

# Consultar
print("\nAlumnos con asistencia:")
print(list(db.alumnos.find({"asistencia": True})))

---
# 🧪 Laboratorio Comparativo: MongoDB vs Redis vs PostgreSQL

## Estructura comparativa

| Base de datos | Tipo | Estructura | Ideal para |
|---------------|------|------------|------------|
| MongoDB | NoSQL (documentos) | JSON-like, flexible | Datos semi-estructurados, consultas dinámicas |
| Redis | NoSQL (clave-valor) | Strings, hashes, listas, sets | Acceso ultrarrápido, caché, contadores |
| PostgreSQL | SQL relacional | Tablas con esquema fijo | Consultas complejas, integridad de datos |

## 📊 Dataset base

In [ ]:
usuarios = [
    {"id": 1, "nombre": "Ana", "edad": 28, "activo": True, "fecha_registro": "2023-01-15"},
    {"id": 2, "nombre": "Luis", "edad": 35, "activo": False, "fecha_registro": "2022-11-03"},
    {"id": 3, "nombre": "Carlos", "edad": 42, "activo": True, "fecha_registro": "2023-06-21"},
    {"id": 4, "nombre": "Sofía", "edad": 30, "activo": True, "fecha_registro": "2023-03-10"}
]

print("Dataset cargado con", len(usuarios), "usuarios")

## 🟥 Insertar en REDIS

Carga como hashes + conjuntos

In [ ]:
import redis

r = redis.Redis(host='redis', port=6379, decode_responses=True)

# Convertir valores a tipos compatibles con Redis
def convertir_para_redis(user):
    return {
        "id": str(user["id"]),
        "nombre": user["nombre"],
        "edad": str(user["edad"]),
        "activo": str(int(user["activo"])),  # True → "1", False → "0"
        "fecha_registro": user["fecha_registro"]
    }

# Cargar usuarios como hashes
for user in usuarios:
    r.hset(f"usuario:{user['id']}", mapping=convertir_para_redis(user))

# Crear conjunto de IDs activos
r.delete("usuarios_activos")
for user in usuarios:
    if user["activo"]:
        r.sadd("usuarios_activos", user["id"])

print("✅ Datos cargados en Redis")
print("Usuarios activos (IDs):", r.smembers("usuarios_activos"))

## 🟦 Insertar en POSTGRESQL

Carga en tabla relacional

In [ ]:
import psycopg2

conn_pg = psycopg2.connect(
    host="postgres",
    port=5432,
    user="admin",
    password="admin",
    dbname="testdb"
)
cursor_pg = conn_pg.cursor()

# Crear tabla usuarios
cursor_pg.execute("""
CREATE TABLE IF NOT EXISTS usuarios (
    id SERIAL PRIMARY KEY,
    nombre TEXT,
    edad INT,
    activo BOOLEAN,
    fecha_registro DATE
)
""")
conn_pg.commit()

# Limpiar tabla para evitar duplicados
cursor_pg.execute("DELETE FROM usuarios")
conn_pg.commit()

# Insertar usuarios
for user in usuarios:
    cursor_pg.execute(
        "INSERT INTO usuarios (id, nombre, edad, activo, fecha_registro) VALUES (%s, %s, %s, %s, %s)",
        (user["id"], user["nombre"], user["edad"], user["activo"], user["fecha_registro"])
    )
conn_pg.commit()

# Verificar
cursor_pg.execute("SELECT * FROM usuarios")
print("✅ Datos cargados en PostgreSQL")
print(cursor_pg.fetchall())

## 🟩 Insertar en MONGODB

Insertar documentos JSON

In [ ]:
from pymongo import MongoClient

client_mongo = MongoClient("mongodb://mongodb:27017/")
db_mongo = client_mongo.bigdata

# Limpiar colección para evitar duplicados
db_mongo.usuarios.delete_many({})

# Insertar usuarios
db_mongo.usuarios.insert_many(usuarios)

print("✅ Datos cargados en MongoDB")
print(list(db_mongo.usuarios.find()))

---
# 📈 Consulta Comparativa

## 🧪 Consulta objetivo
**Filtrar usuarios activos con edad ≥ 30**

1. Ejecutar consulta equivalente en MongoDB, Redis y PostgreSQL
2. Medir tiempo de ejecución
3. Mostrar resultados en gráfico comparativo

In [ ]:
import time
import matplotlib.pyplot as plt
import redis
from pymongo import MongoClient
import psycopg2

# ========================
# 🍃 MongoDB
# ========================
client_mongo = MongoClient("mongodb://mongodb:27017/")
db_mongo = client_mongo.bigdata

start = time.time()
mongo_result = list(db_mongo.usuarios.find({"activo": True, "edad": {"$gte": 30}}))
mongo_time = time.time() - start

print("MongoDB - Usuarios activos con edad >= 30:")
for u in mongo_result:
    print(f"  - {u['nombre']}, edad: {u['edad']}")

# ========================
# 🔴 Redis
# ========================
r = redis.Redis(host='redis', port=6379, decode_responses=True)

def reconstruir_usuario(uid):
    raw = r.hgetall(f"usuario:{uid}")
    return {
        "id": int(raw["id"]),
        "nombre": raw["nombre"],
        "edad": int(raw["edad"]),
        "activo": bool(int(raw["activo"])),
        "fecha_registro": raw["fecha_registro"]
    }

start = time.time()
usuarios_ids = r.smembers("usuarios_activos")
redis_result = [reconstruir_usuario(uid) for uid in usuarios_ids if int(r.hget(f"usuario:{uid}", "edad")) >= 30]
redis_time = time.time() - start

print("\nRedis - Usuarios activos con edad >= 30:")
for u in redis_result:
    print(f"  - {u['nombre']}, edad: {u['edad']}")

# ========================
# 🐘 PostgreSQL
# ========================
conn_pg = psycopg2.connect(
    host="postgres",
    port=5432,
    user="admin",
    password="admin",
    dbname="testdb"
)
cursor_pg = conn_pg.cursor()

start = time.time()
cursor_pg.execute("SELECT * FROM usuarios WHERE activo = TRUE AND edad >= 30")
postgres_result = cursor_pg.fetchall()
postgres_time = time.time() - start

print("\nPostgreSQL - Usuarios activos con edad >= 30:")
for u in postgres_result:
    print(f"  - {u[1]}, edad: {u[2]}")

# ========================
# Resumen de tiempos
# ========================
print("\n" + "="*50)
print("⏱️ TIEMPOS DE EJECUCIÓN")
print("="*50)
print(f"MongoDB:    {mongo_time:.6f} segundos")
print(f"Redis:      {redis_time:.6f} segundos")
print(f"PostgreSQL: {postgres_time:.6f} segundos")

## 📊 Visualización Comparativa

In [ ]:
import matplotlib.pyplot as plt

# Datos para el gráfico
labels = ['MongoDB', 'Redis', 'PostgreSQL']
times = [mongo_time, redis_time, postgres_time]
colors = ['#4DB33D', '#DC382D', '#336791']  # Colores oficiales de cada BD

# Crear gráfico
plt.figure(figsize=(10, 6))
bars = plt.bar(labels, times, color=colors, edgecolor='black', linewidth=1.2)

# Configurar etiquetas y título
plt.ylabel('Tiempo de ejecución (segundos)', fontsize=12)
plt.xlabel('Base de datos', fontsize=12)
plt.title('🧪 Comparación de Rendimiento: MongoDB vs Redis vs PostgreSQL\n(Consulta: usuarios activos con edad ≥ 30)', fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Mostrar tiempo sobre cada barra
for bar, t in zip(bars, times):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0001, 
             f"{t:.6f}s", ha='center', va='bottom', fontsize=11, fontweight='bold')

# Añadir leyenda con tipo de storage
plt.figtext(0.15, 0.02, '🍃 MongoDB: Documental | 🔴 Redis: En Memoria | 🐘 PostgreSQL: Relacional', 
            ha='left', fontsize=10, style='italic')

plt.tight_layout()
plt.savefig('comparacion_storages.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Gráfico guardado como 'comparacion_storages.png'")

---
## 📝 Conclusiones

| Característica | MongoDB | Redis | PostgreSQL |
|----------------|---------|-------|------------|
| **Tipo** | NoSQL Documental | NoSQL Key-Value | SQL Relacional |
| **Velocidad** | Rápido para consultas flexibles | Ultra rápido (en memoria) | Optimizado para consultas complejas |
| **Esquema** | Flexible (sin esquema) | Sin esquema | Esquema fijo |
| **Persistencia** | Disco | Memoria (opcional disco) | Disco |
| **Ideal para** | Datos semi-estructurados | Caché, sesiones, contadores | Datos relacionales, transacciones |